In [1]:
!nvidia-smi

Mon Apr 22 14:32:54 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 470.103.01   Driver Version: 470.103.01   CUDA Version: 11.4     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  Tesla K80           Off  | 00000000:05:00.0 Off |                    0 |
| N/A   33C    P8    26W / 149W |     27MiB / 11441MiB |      0%      Default |
|                               |                      |                  N/A |
+-------------------------------+----------------------+----------------------+
|   1  Tesla K80           Off  | 00000000:06:00.0 Off |                    0 |
| N/A   

# Import Lib

In [2]:
import os
import nibabel as nib
import numpy as np 
from collections import OrderedDict
import json
from pathlib import Path

from utils.helper import convert_nrrd_to_nifti, create_folder, plot_all_slices, plot_histogram, generate_binary_image, adjust_affine_for_spacing_and_origin, save_binary_image_with_adjusted_origin, make_if_dont_exist
from utils.metrics import dice_score_per_class, hausdorff_distance_per_class, ravd_per_class

## Path setup

In [3]:

# define dataset path
BASE_PATH = Path('./').resolve()
DATA_PATH = BASE_PATH / 'dataset'
ORG_DATA_PATH = Path('/work/shared/ngmm/3Dimage/Analysed_brains/Wdr47Kusss').resolve()
ORG_TEST_DATA_PATH = Path('/work/shared/ngmm/3Dimage/Analysed_brains/Zbtb20').resolve()
ORG_SEG_DATA_PATH = Path('/work/shared/ngmm/scripts/Taiabur/seg.data').resolve()

project_name = 'TRF' #change here for different task name
task_name = 'Dataset004_' + project_name 

TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTr'
GT_TRAINING_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTr'
TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'imagesTs'
GT_TEST_DATASET_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name / 'labelsTs'
PREDICTION_RESULTS_PATH  = BASE_PATH / 'dataset/nnUNet_Prediction_Results' / task_name
TASK_PATH = BASE_PATH / 'dataset/nnUNet_raw_data' / task_name 

# setup environment variables
nnUNet_raw = BASE_PATH / 'dataset/nnUNet_raw_data'
nnUNet_preprocessed = BASE_PATH / 'dataset/nnUNet_preprocessed'
nnUNet_results = BASE_PATH / 'dataset/nnUNet_results'

In [ ]:
make_if_dont_exist(TRAINING_DATASET_PATH,overwrite=False)
make_if_dont_exist(GT_TRAINING_DATASET_PATH)
make_if_dont_exist(TEST_DATASET_PATH)
make_if_dont_exist(GT_TEST_DATASET_PATH)
make_if_dont_exist(PREDICTION_RESULTS_PATH)

make_if_dont_exist(nnUNet_preprocessed)
make_if_dont_exist(nnUNet_results)

## nnU-Net v2 installtion 

In [2]:
pip install nnunetv2

  Using cached nnunetv2-2.4.2.tar.gz (184 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached torch-2.3.1-cp312-cp312-manylinux1_x86_64.whl.metadata (26 kB)
  Using cached acvl_utils-0.2.tar.gz (18 kB)
  Preparing metadata (setup.py) ... done
  Using cached dynamic_network_architectures-0.3.1.tar.gz (20 kB)
  Preparing metadata (setup.py) ... done
  Using cached tqdm-4.66.4-py3-none-any.whl.metadata (57 kB)
  Using cached dicom2nifti-2.4.11-py3-none-any.whl.metadata (1.3 kB)
  Using cached scipy-1.13.1-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (60 kB)
  Using cached batchgenerators-0.25.tar.gz (61 kB)
  Preparing metadata (setup.py) ... done
  Using cached numpy-1.26.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (61 kB)
  Using cached scikit_learn-1.5.0-cp312-cp312-manylinux_2_17_x86_64.manylin

### Clone nnU-Net code

In [ ]:
!git clone https://github.com/MIC-DKFZ/nnUNet.git

### Install hiddenlayer

In [ ]:
pip install --upgrade git+https://github.com/FabianIsensee/hiddenlayer.git

In [ ]:
os.chdir(BASE_PATH)
os.getcwd()

In [ ]:
os.chdir('nnUNet')
os.getcwd()

In [ ]:
pip install -e .

In [ ]:
os.chdir(BASE_PATH)
os.getcwd()

### train dataset: convert image .nrrd to nifti format

In [ ]:
# over view original Images 
all_files = os.listdir(ORG_DATA_PATH)
for file in all_files:
    if file.endswith("_RCL5.nrrd"):
        source_nrrd_file_path = os.path.join(ORG_DATA_PATH, file)
        save_nifti_file_path = os.path.join(TRAINING_DATASET_PATH, file)
        # Automatically generate the nifti_file_path based on nrrd_file_path
        base_name = os.path.splitext(os.path.basename(save_nifti_file_path))[0]
        save_nifti_file_path = os.path.join(os.path.dirname(save_nifti_file_path), base_name + '_0000.nii.gz')
        # print(source_nrrd_file_path)
        # print(save_nifti_file_path)
        convert_nrrd_to_nifti(source_nrrd_file_path, save_nifti_file_path)
print('File convert done. ')

### test dataset: convert image .nrrd to nifti format

In [ ]:
# over view original Images 
all_files = os.listdir(ORG_TEST_DATA_PATH)
for file in all_files:
    if file.endswith("_RCL5.nrrd"):
        source_nrrd_file_path = os.path.join(ORG_TEST_DATA_PATH, file)
        save_nifti_file_path = os.path.join(TEST_DATASET_PATH, file)
        # Automatically generate the nifti_file_path based on nrrd_file_path
        base_name = os.path.splitext(os.path.basename(save_nifti_file_path))[0]
        save_nifti_file_path = os.path.join(os.path.dirname(save_nifti_file_path), base_name + '_0000.nii.gz')
        # print(source_nrrd_file_path)
        # print(save_nifti_file_path)
        convert_nrrd_to_nifti(source_nrrd_file_path, save_nifti_file_path)
print('File convert done. ')

### label image generate 

In [ ]:
# Loop through each file in the directory
for file in os.listdir(ORG_SEG_DATA_PATH):
    if file.endswith("brainmask.nii"):
        file_path = ORG_SEG_DATA_PATH / file
        img = nib.load(file_path)
        data = img.get_fdata()
        print(f"Processing {file}...")
        
        # Calculate a simple threshold (mean intensity value)
        threshold = np.mean(data)
        # Generate binary image
        binary_data = generate_binary_image(data, threshold)
        # Save binary image
        binary_file_path = GT_TRAINING_DATASET_PATH / file
        base_name = os.path.splitext(os.path.basename(binary_file_path))[0]
        binary_file_path = GT_TRAINING_DATASET_PATH / base_name
        save_binary_image_with_adjusted_origin(binary_data, img, str(binary_file_path))
        print(f"Binary image saved as {binary_file_path}")
        

        # Optionally, plot original and binary image slices for verification
        plot_all_slices(data)  # Original
        plot_all_slices(binary_data)  # Binary

### image name check and rename

In [ ]:
import os
import re

# Function to rename the files
def rename_images(directory):
    # Define the regex pattern for the original filenames (adjust if necessary)
    pattern = re.compile(r'NG(\d+)_RCL5_(\d+)\.nii\.gz')

    # Counter for renamed files
    renamed_files = 0

    # List all files in the directory
    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            # Construct the new filename
            new_filename = f'{project_name}_NG_{match.group(1)}_{match.group(2)}.nii.gz'
            # Construct the full old and new file paths
            old_file = os.path.join(directory, filename)
            new_file = os.path.join(directory, new_filename)
            # Rename the file
            os.rename(old_file, new_file)
            renamed_files += 1
            print(f'Renamed: {filename} -> {new_filename}')
    
    return renamed_files

# Run the renaming function and print the number of files renamed
num_renamed = rename_images(TRAINING_DATASET_PATH)
num_renamed


### label name check and rename

In [ ]:

# Function to rename the files for the second task
def rename_images_second_task(directory):
    # Define the regex pattern for the original filenames (adjust if necessary)
    pattern = re.compile(r'NG(\d+)_brainmask\.nii\.gz')

    # Counter for renamed files
    renamed_files = 0
    # List all files in the directory
    for filename in os.listdir(directory):
        match = pattern.match(filename)
        if match:
            # Construct the new filename
            new_filename = f'Wdr47Kusss_NG_{match.group(1)}.nii.gz'
            # Construct the full old and new file paths
            old_file = os.path.join(directory, filename)
            new_file = os.path.join(directory, new_filename)
            # Rename the file
            os.rename(old_file, new_file)
            renamed_files += 1
            print(f'Renamed: {filename} -> {new_filename}')
    
    return renamed_files

# Run the renaming function for the second task and print the number of files renamed
num_renamed_2 = rename_images_second_task(GT_TRAINING_DATASET_PATH)
num_renamed_2

In [ ]:
train_files = os.listdir(TRAINING_DATASET_PATH)
label_files = os.listdir(GT_TRAINING_DATASET_PATH)
print("train image files:",len(train_files))
print("train label files:",len(label_files))
print("Matches:",len(set(train_files).intersection(set(label_files))))


In [ ]:
print("Testing files:",len(os.listdir(TEST_DATASET_PATH)))
print("Testing label files:",len(os.listdir(GT_TEST_DATASET_PATH)))
print(TEST_DATASET_PATH)
print(GT_TEST_DATASET_PATH)

### generate dataset.json file

In [ ]:
from typing import Tuple
import json
from os.path import join

def save_json(data, file_path, sort_keys=False):

    with open(file_path, 'w') as f:
        json.dump(data, f, indent=4, sort_keys=sort_keys)

def generate_dataset_json(output_folder: str,
                          channel_names: dict,
                          labels: dict,
                          num_training_cases: int,
                          file_ending: str,
                          regions_class_order: Tuple[int, ...] = None,
                          dataset_name: str = None, reference: str = None, release: str = None, license: str = None,
                          description: str = None,
                          overwrite_image_reader_writer: str = None, **kwargs):
    
    has_regions: bool = any([isinstance(i, (tuple, list)) and len(i) > 1 for i in labels.values()])
    if has_regions:
        assert regions_class_order is not None, f"You have defined regions but regions_class_order is not set. " \
                                                f"You need that."
    # channel names need strings as keys
    keys = list(channel_names.keys())
    for k in keys:
        if not isinstance(k, str):
            channel_names[str(k)] = channel_names[k]
            del channel_names[k]

    # labels need ints as values
    for l in labels.keys():
        value = labels[l]
        if isinstance(value, (tuple, list)):
            value = tuple([int(i) for i in value])
            labels[l] = value
        else:
            labels[l] = int(labels[l])

    dataset_json = {
        'channel_names': channel_names,  # previously this was called 'modality'. I didn't like this so this is
        # channel_names now. Live with it.
        'labels': labels,
        'numTraining': num_training_cases,
        'file_ending': file_ending,
    }

    if dataset_name is not None:
        dataset_json['name'] = dataset_name
    if reference is not None:
        dataset_json['reference'] = reference
    if release is not None:
        dataset_json['release'] = release
    if license is not None:
        dataset_json['licence'] = license
    if description is not None:
        dataset_json['description'] = description
    if overwrite_image_reader_writer is not None:
        dataset_json['overwrite_image_reader_writer'] = overwrite_image_reader_writer
    if regions_class_order is not None:
        dataset_json['regions_class_order'] = regions_class_order

    dataset_json.update(kwargs)

    save_json(dataset_json, join(output_folder, 'dataset.json'), sort_keys=False)
    

# List all files in the training images and labels directories
image_files = os.listdir(TRAINING_DATASET_PATH)
label_files = os.listdir(GT_TRAINING_DATASET_PATH)
test_ids = os.listdir(TEST_DATASET_PATH)

channel_names = {"0": "microscopic"}
num_training_cases = len(image_files)  
file_ending = ".nii.gz"

generate_dataset_json(
    output_folder=TASK_PATH,
    channel_names=channel_names,
    labels={"background":0, "Total Brain":1},
    num_training_cases=num_training_cases,
    file_ending=file_ending,
    dataset_name="Mouse Brain Segmentation",
    description="Mouse Brain Segmentation",
    reference="",
    release="0.0",
    license="",
    training = [{'image': f"./imagesTr/{image_file}", 'label': f"./labelsTr/{label_file}"} 
            for image_file, label_file in zip(sorted(image_files), sorted(label_files))],
    test=["./imagesTs/%s" % (i[:i.find("_0000")] + '.nii.gz') for i in test_ids]
)


In [ ]:
print('dataset.json file generated.')

## set up environment variables

Setup check ``{echo $nnUNet_raw, echo $nnUNet_preprocessed, echo $nnUNet_results}``


In [4]:
%env nnUNet_raw=$nnUNet_raw
%env nnUNet_preprocessed=$nnUNet_preprocessed
%env nnUNet_results=$nnUNet_results

env: nnUNet_raw=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data
env: nnUNet_preprocessed=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_preprocessed
env: nnUNet_results=/beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results


In [ ]:
!nnUNetv2_plan_and_preprocess -d 1 --verify_dataset_integrity

### 2d

In [ ]:
!nnUNetv2_train 1 2d 0 --npz

In [ ]:
!nnUNetv2_train 1 2d 1 --npz

In [ ]:
!nnUNetv2_train 1 2d 2 --npz

In [ ]:
!nnUNetv2_train 1 2d 3 --npz

In [ ]:
!nnUNetv2_train 1 2d 4 --npz

### 3d_fullres

In [ ]:
!nnUNetv2_train 1 3d_fullres 0 --npz

In [ ]:
!nnUNetv2_train 1 3d_fullres 1 --npz

In [ ]:
!nnUNetv2_train 1 3d_fullres 2 --npz

In [ ]:
!nnUNetv2_train 1 3d_fullres 3 --npz

In [ ]:
!nnUNetv2_train 1 3d_fullres 4 --npz

### 3d_lowres

In [ ]:
!nnUNetv2_train 1 3d_lowres 0 --npz 

In [ ]:
!nnUNetv2_train 1 3d_lowres 1 --npz 

In [ ]:
!nnUNetv2_train 1 3d_lowres 2 --npz 

In [ ]:
!nnUNetv2_train 1 3d_lowres 3 --npz 

In [ ]:
!nnUNetv2_train 1 3d_lowres 4 --npz 

## 2d

### Best configuration check

In [ ]:
!nnUNetv2_find_best_configuration 1 -c 2d 


### prediction

In [ ]:
!nnUNetv2_predict -d 1 -i /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data -o /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/2d -f  0 1 2 3 4 -tr nnUNetTrainer -c 2d -p nnUNetPlans

### post processing

In [ ]:
!nnUNetv2_apply_postprocessing -i /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss -o /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_postprocessed/1_18 -pp_pkl_file /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset001_Wdr47Kusss/nnUNetTrainer__nnUNetPlans__2d/crossval_results_folds_0_1_2_3_4/postprocessing.pkl -np 8 -plans_json /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset001_Wdr47Kusss/nnUNetTrainer__nnUNetPlans__2d/crossval_results_folds_0_1_2_3_4/plans.json

## 3d_lowres

### find best config

In [ ]:
!nnUNetv2_find_best_configuration 1 -c 3d_lowres 

### prediction

In [ ]:
!nnUNetv2_predict -d Dataset001_Wdr47Kusss -i /work/shared/ngmm/scripts/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data -o /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_final_result/3d_lowres -f  0 1 2 3 4 -tr nnUNetTrainer -c 3d_lowres -p nnUNetPlans

### post processing

In [ ]:
!nnUNetv2_apply_postprocessing -i /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/3d_lowres -o /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_postprocessed/3d_lowres -pp_pkl_file /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset001_Wdr47Kusss/nnUNetTrainer__nnUNetPlans__3d_lowres/crossval_results_folds_0_1_2_3_4/postprocessing.pkl -np 8 -plans_json /beegfs/data/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset001_Wdr47Kusss/nnUNetTrainer__nnUNetPlans__3d_lowres/crossval_results_folds_0_1_2_3_4/plans.json

## 3d_fullres

### find best config

In [ ]:
!nnUNetv2_find_best_configuration 1 -c 3d_fullres 

### prediction

In [ ]:
!nnUNetv2_predict -d 1 -i /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data -o /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/3d_fullres -f  0 1 2 3 4 -tr nnUNetTrainer -c 3d_fullres -p nnUNetPlans

### postprocessing

In [ ]:
nnUNetv2_apply_postprocessing -i /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/3d_fullres -o /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_postprocessed/3d_fullres -pp_pkl_file /work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset001_Wdr47Kusss/nnUNetTrainer__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4/postprocessing.pkl -np 8 -plans_json /user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset001_Wdr47Kusss/nnUNetTrainer__nnUNetPlans__3d_fullres/crossval_results_folds_0_1_2_3_4/plans.json

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt


def image_plot(original_img_path, processed_img_path, image_id):
    original_img = nib.load(original_img_path)
    processed_img = nib.load(processed_img_path)

    # Convert NIfTI images to numpy arrays
    original_data = original_img.get_fdata()
    processed_data = processed_img.get_fdata()

    processed_data = original_data * processed_data
    
    origina_shape = original_data.shape
    processed_shape = processed_data.shape

    # Assuming the images are 3D, we take the middle slice of the first dimension for demonstration
    mid_slice = original_data.shape[0] // 2
    slice = 210
    fig, axes = plt.subplots(1, 2, figsize=(12, 10))
    axes[0].imshow(original_data[:, :, slice], cmap='gray')
    axes[0].set_title(f'Original: {image_id} {origina_shape} , slice: {slice}')
    axes[0].axis('off')  # Hide axes for clarity

    axes[1].imshow(processed_data[:, :,slice], cmap='gray')
    axes[1].set_title(f'Processed: {image_id} {processed_shape}, slice: {slice}')
    axes[1].axis('off')  # Hide axes for clarity

    plt.show()

In [ ]:
image_name = 'NG_2617'
# Load the original and processed NIfTI images
original_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_raw_data/Dataset001_Wdr47Kusss/imagesTr/Wdr47Kusss_NG_2617_0000.nii.gz'
processed_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_results/Dataset001_Wdr47Kusss/nnUNetTrainer__nnUNetPlans__2d/fold_2/validation/Wdr47Kusss_NG_2617.nii.gz'
image_plot(original_img_path, processed_img_path, image_name)

In [ ]:
image_name = 'NG4126'
# Load the original and processed NIfTI images
original_img_path = 'dataset/nnUNet_test_data/time/NG4126_RCL5_0000.nii.gz'
processed_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/2d/NG4126_RCL5.nii.gz'
image_plot(original_img_path, processed_img_path, image_name)

In [ ]:
image_name = 'NG4127'
# Load the original and processed NIfTI images
original_img_path = 'dataset/nnUNet_test_data/time/NG4127_RCL5_0000.nii.gz'
processed_img_path = '/work/shared/ngmm/scripts/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/2d/NG4127_RCL5.nii.gz'
image_plot(original_img_path, processed_img_path, image_name)

In [ ]:
image_name = 'NG4126'
# Load the original and processed NIfTI images
original_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data/NG4126_RCL5_0000.nii.gz'
processed_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/3d/NG4126_RCL5.nii.gz'
image_plot(original_img_path, processed_img_path, image_name)

In [ ]:
image_name = 'NG4127'
# Load the original and processed NIfTI images
original_img_path = 'dataset/nnUNet_test_data/NG4127_RCL5_0000.nii.gz'
processed_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/3d/NG4127_RCL5.nii.gz'
image_plot(original_img_path, processed_img_path, image_name)

In [ ]:

image_name = 'NG4127'
# Load the original and processed NIfTI images
original_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data/time/NG4110.nii.gz'
processed_img_path = '/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/3d/NG4127_RCL5.nii.gz'
image_plot(original_img_path, processed_img_path, image_name)

In [ ]:
import nibabel as nib
import matplotlib.pyplot as plt
import numpy as np

def show_image_and_histogram(nifti_file_path, slice_index):
    # Load the NIfTI file
    img = nib.load(nifti_file_path)
    data = img.get_fdata()
    
    # Select a slice (in this case, an axial slice)
    slice_ = data[:, :, slice_index]
    
    # Create a figure with 2 columns: one for the image, one for the histogram
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))
    
    # Display the image slice
    axes[0].imshow(slice_.T, cmap='gray', origin='lower')
    axes[0].set_title('Slice {}'.format(slice_index))
    
    # Display the histogram of the slice
    # We use `.ravel()` to flatten the array for histogram computation
    axes[1].hist(slice_.ravel(), bins=100, color='c')
    axes[1].set_title('Histogram of Intensity Values')
    
    # Enhance layout
    plt.tight_layout()
    plt.show()

# Example usage
nifti_file_path = '/work/shared/ngmm/scripts/Taiabur/NG2564_SegmentsSeeds.nii' # Replace with the path to your NIfTI file
slice_index = 190 # Choose the slice index you are interested in
show_image_and_histogram(nifti_file_path, slice_index)


In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# Load NIfTI file
# nifti_file_path = 'your_image.nii'  # Replace with your file path
img = nib.load(nifti_file_path)
data = img.get_fdata()
print(data.shape)
# Sample the data to reduce size - for visualization purposes
# You can adjust the sampling rate based on the size of your data and your system's capabilities
subsampled_data = data[::10, ::10, ::10]

# Get the indices where the data is above a threshold to reduce points
x, y, z = np.where(subsampled_data > np.percentile(subsampled_data, 90))  # Adjust the percentile as needed

# Use intensity values as colors
colors = subsampled_data[x, y, z]

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Create a scatter plot
scat = ax.scatter(x, y, z, c=colors, cmap='viridis')

# Colorbar to show intensity scale
plt.colorbar(scat, shrink=0.5, aspect=5, label='Intensity')

# Labeling
ax.set_xlabel('X')
ax.set_ylabel('Y')
ax.set_zlabel('Z')
plt.title('3D Scatter Plot of Voxel Intensities')

plt.show()


In [ ]:
import nibabel as nib
import numpy as np

def calculate_roi_intensities(image_path, label_path):
    # Load the original image and the label image
    image = nib.load(image_path).get_fdata()
    labels = nib.load(label_path).get_fdata()
    
    # Find unique labels/ROIs in the segmented label volume
    unique_labels = np.unique(labels)
    
    # Dictionary to hold the average intensity for each ROI
    roi_intensities = {}
    
    # Calculate the average intensity for each ROI
    for label in unique_labels:
        if label == 0:
            continue  # Assuming label 0 is the background or not of interest
        mask = labels == label
        roi_intensity = np.mean(image[mask])
        roi_intensities[label] = roi_intensity
        print(f"ROI {int(label)}: Average Intensity = {roi_intensity}")
    
    return roi_intensities


# Example usage
image_path = '/work/shared/ngmm/scripts/Taiabur/r24/NG2564_SegmentsSeeds.nii'
label_path = '/work/shared/ngmm/scripts/Taiabur/r24/NG2564_SegmentsSeeds.nii'
roi_intensities = calculate_roi_intensities(image_path, label_path)

In [ ]:
# Example usage
image_path = '/work/shared/ngmm/scripts/Taiabur/r24/NG2561_Segments_Elo.nii'
label_path = '/work/shared/ngmm/scripts/Taiabur/r24/NG2561_Segments_Elo.nii'
roi_intensities = calculate_roi_intensities(image_path, label_path)

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt

def modify_and_save_roi_slices(image_path, roi_slices, new_intensity, output_prefix):
    """
    Modify intensity values inside the ROI for specific slices and save them as images.

    Parameters:
    - image_path: Path to the NIfTI image file.
    - roi_slices: List of slice indices to be modified.
    - new_intensity: The new intensity value to be assigned within the ROI.
    - output_prefix: Prefix for the output image files.
    """
    # Load the image
    img = nib.load(image_path)
    data = img.get_fdata()

    # Loop through the specified slices
    for i, slice_idx in enumerate(roi_slices):
        # Copy the slice to avoid modifying the original data
        slice_data = np.copy(data[:, :, slice_idx])
        
        # Assuming the ROI is defined by non-zero values
        roi_mask = slice_data > 0
        
        # Modify the intensity within the ROI
        slice_data[roi_mask] = new_intensity
        
        # Save the modified slice as an image
        plt.imshow(slice_data.T, cmap='gray', origin='lower')
        plt.axis('off')  # Remove axis for clarity
        output_path = f"{output_prefix}_slice_{slice_idx}.png"
        plt.savefig(output_path, bbox_inches='tight', pad_inches=0)
        plt.close()  # Close the plot to free memory
        print(f"Saved modified slice {slice_idx} as {output_path}")

# Example usage
image_path = '/work/shared/ngmm/scripts/Taiabur/NG2564_SegmentsSeeds.nii'
roi_slices = [20, 40, 60]  # Indices of slices to modify
new_intensity = 500  # New intensity value for the ROI
output_prefix = 'modified_roi'
modify_and_save_roi_slices(image_path, roi_slices, new_intensity, output_prefix)


In [ ]:
import nibabel as nib
import numpy as np
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path = '/work/shared/ngmm/scripts/Taiabur/r24/NG2564_SegmentsSeeds.nii'
img = nib.load(nifti_path)
data = img.get_fdata()

# Function to display a slice
def display_slice(slice_no):
    plt.figure(figsize=(8, 8))
    plt.imshow(data[:, :, slice_no], cmap='gray')
    plt.axis('off')
    plt.show()

# Interactive widget
slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

widgets.interactive(display_slice, slice_no=slice_slider)


In [1]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path = '/work/shared/ngmm/scripts/Taiabur/r24/NG2564_SegmentsSeeds.nii'
# nifti_path = '/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/NG2561_RCL5.nii.gz'
img = nib.load(nifti_path)
data = img.get_fdata()

# Adjusted function to display a slice and its histogram
def display_slice_and_histogram(slice_no):
    # Setup the figure and axes for a side-by-side plot: slice and histogram
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Display the slice
    ax = axes[0]
    ax.imshow(data[:, :, slice_no], cmap='gray')
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no}')
    
    # Display the histogram
    ax = axes[1]
    slice_data = data[:, :, slice_no].ravel()
    ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    ax.set_title('Pixel Intensity Distribution')
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

# Interactive widget for slice selection
slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

widgets.interactive(display_slice_and_histogram, slice_no=slice_slider)


FileNotFoundError: No such file or no access: '/work/shared/ngmm/scripts/Taiabur/r24/NG2564_SegmentsSeeds.nii'

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path = '/work/shared/ngmm/scripts/Taiabur/r24/NG2563_Segments_Elo2.nii'
# nifti_path = '/user1/ngmm/tr855969/Desktop/Taiabur/ngmm-nnunet/dataset/nnUNet_Prediction_Results/Dataset001_Wdr47Kusss/NG2561_RCL5.nii.gz'
img = nib.load(nifti_path)
data = img.get_fdata()

# Adjusted function to display a slice and its histogram
def display_slice_and_histogram(slice_no):
    # Setup the figure and axes for a side-by-side plot: slice and histogram
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Display the slice
    ax = axes[0]
    ax.imshow(data[:, :, slice_no], cmap='gray')
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no}')
    
    # Display the histogram
    ax = axes[1]
    slice_data = data[:, :, slice_no].ravel()
    ax.hist(slice_data, bins=50, color='c', alpha=0.75)
    ax.set_title('Pixel Intensity Distribution')
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

# Interactive widget for slice selection
slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

widgets.interactive(display_slice_and_histogram, slice_no=slice_slider)


In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

# Load the NIfTI file
nifti_path = '/work/shared/ngmm/scripts/Taiabur/NG2564_SegmentsSeeds.nii'
img = nib.load(nifti_path)
data = img.get_fdata()

def display_slice_and_column_histogram(slice_no, column_no):
    # Setup the figure and axes for a two-subplot layout: one for the slice and one for the histogram
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    
    # Display the slice
    ax = axes[0]
    ax.imshow(data[:, :, slice_no], cmap='gray', aspect='auto')
    ax.axvline(x=column_no, color='r', linestyle='--')  # Mark the column
    ax.axis('off')  # Hide axes ticks
    ax.set_title(f'Slice {slice_no}, Column {column_no}')
    
    # Display the histogram for the column across all slices
    ax = axes[1]
    column_data = data[:, column_no, :].ravel()  # Extract all pixel values for the column across slices
    ax.hist(column_data, bins=50, color='c', alpha=0.75)
    ax.xticks(range(1, 1001, 50))
    ax.set_title('Column Intensity Distribution')
    ax.grid(True)
    
    plt.tight_layout()
    plt.show()

# Interactive widgets
slice_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[2] - 1, 
    step=1, 
    value=data.shape[2] // 2, 
    description='Slice'
)

column_slider = widgets.IntSlider(
    min=0, 
    max=data.shape[1] - 1, 
    step=1, 
    value=data.shape[1] // 2, 
    description='Column'
)

widgets.interactive(display_slice_and_column_histogram, slice_no=slice_slider, column_no=column_slider)


In [ ]:
import os
from tabulate import tabulate

def process_test_volumes(test_vol_path, test_vol_gt_path):
    test_volume_dir = os.listdir(test_vol_path)

    rows = []
    for file in test_volume_dir:
        if file.endswith(".nii.gz"):
            # Construct the full path to the test volume
            test_vol = os.path.join(test_vol_path, file)

            # Extract the base name without the extension
            base_name = os.path.splitext(file)[0]
  
            # Construct the ID using the first two parts of the base name
            image_id = f'{base_name.split("_")[0]}_{base_name.split("_")[1]}_'
            
            # Construct the full path to the corresponding ground truth volume
            gt_vol = os.path.join(test_vol_gt_path, image_id + "masked_0000.nii.gz")
        
            rows.append((base_name.split("_")[0],) +tuple(dice_score_per_class(nib.load(test_vol).get_fdata(), nib.load(gt_vol).get_fdata())))
    return rows
    
# Example usage
test_vol_path = "/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data/q"
test_vol_gt_path = "/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data_gt"
ds_score = process_test_volumes(test_vol_path, test_vol_gt_path)

print(tabulate(ds_score, headers=['Image id','Background', 'TB'], tablefmt="grid"))

In [ ]:
import os
import nibabel as nib
from tabulate import tabulate

def process_test_volumes(test_vol_path, test_vol_gt_path):
    test_volume_dir = os.listdir(test_vol_path)

    rows = []
    for file in test_volume_dir:
        if file.endswith(".nii.gz"):
            # Construct the full path to the test volume
            test_vol = os.path.join(test_vol_path, file)
            
            # Extract the base name without the extension (twice to remove .nii.gz)
            base_name = os.path.splitext(os.path.splitext(file)[0])[0]

            # Construct the ID using the first two parts of the base name
            image_id = f'{base_name.split("_")[0]}_{base_name.split("_")[1]}'

            # Construct the full path to the corresponding ground truth volume
            gt_vol = os.path.join(test_vol_gt_path, image_id + "_masked_0000.nii.gz")

            try:
                # Assuming dice_score_per_class returns a tuple of scores
                scores = dice_score_per_class(nib.load(test_vol).get_fdata(), nib.load(gt_vol).get_fdata())
                rows.append((image_id,) + tuple(scores))  # Append as a tuple
            except Exception as e:
                print(f"Error processing {image_id}: {e}")

    return rows

# Example usage
test_vol_path = "/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data"
test_vol_gt_path = "/work/shared/ngmm/scripts/Taiabur/ngmm-nnunet/dataset/nnUNet_test_data_gt"
ds_score = process_test_volumes(test_vol_path, test_vol_gt_path)

# Printing the table with headers
print(tabulate(ds_score, headers=['Image ID', 'Background', 'TB'], tablefmt="grid"))
